# 01 — Dataset Generation

Generates exactly **300** mixed triangle-inequality prompts (binary + numeric)
and splits them into **train / eval / analysis**.

Outputs written to `my_work/data/`:
- `prompts_triangle.jsonl` — full 300-prompt dataset
- `splits/train_triangle.jsonl`
- `splits/eval.jsonl`
- `splits/analysis.jsonl`

Run this notebook once. No GPU required.

## 0 — Environment Setup

In [1]:
import os
import sys
from pathlib import Path

IN_RUNPOD = os.environ.get("RUNPOD_POD_ID") is not None

def _find_repo_root():
    start = Path.cwd().resolve()
    for directory in [start, *start.parents]:
        if (directory / "circuit_tracer" / "__init__.py").is_file():
            return directory
    workspace = Path("/workspace")
    if workspace.is_dir():
        for child in workspace.iterdir():
            if child.is_dir() and (child / "circuit_tracer" / "__init__.py").is_file():
                return child
    repo_override = os.environ.get("CT_REPO_DIR")
    if repo_override:
        override_path = Path(repo_override).expanduser().resolve()
        if (override_path / "circuit_tracer" / "__init__.py").is_file():
            return override_path
    return None

_root = _find_repo_root()
if _root is not None:
    if str(_root) not in sys.path:
        sys.path.insert(0, str(_root))
    # Add my_work to path so utils/ imports work
    _my_work = _root / "my_work"
    if str(_my_work) not in sys.path:
        sys.path.insert(0, str(_my_work))
    print(f"Repo root: {_root}")
    print(f"my_work  : {_my_work}")
else:
    print("WARNING: could not locate circuit_tracer repo. Set CT_REPO_DIR.")

try:
    from dotenv import load_dotenv
    if _root is not None:
        _env_file = _root / ".env"
        if _env_file.is_file():
            load_dotenv(_env_file)
            print(f"Loaded {_env_file}")
except ImportError:
    pass

# Derive MY_WORK path used by all output cells
MY_WORK = _my_work if _root else Path(".").resolve()
print(f"MY_WORK  : {MY_WORK}")

Repo root: /workspace/thesis_circuit_breaker
my_work  : /workspace/thesis_circuit_breaker/my_work
MY_WORK  : /workspace/thesis_circuit_breaker/my_work


## 1 — Generate 300 prompts

In [2]:
import json
import sys
from collections import Counter

from utils.prompt_generator import generate_triangle_prompts, split_dataset, TOKEN_TRUE, TOKEN_FALSE

SEED = 42
N_PROMPTS = 300

prompts = generate_triangle_prompts(n=N_PROMPTS, seed=SEED)

binary = [p for p in prompts if p.get('task_type', 'binary') == 'binary']
numeric = [p for p in prompts if p.get('task_type', 'binary') == 'numeric']

print(f"Generated {len(prompts)} prompts")
print(f"  Binary : {len(binary)}")
print(f"    True : {sum(1 for p in binary if p['label'])}")
print(f"    False: {sum(1 for p in binary if not p['label'])}")
print(f"  Numeric: {len(numeric)}")
print(f"    Target tokens: {sorted(set(p['label_token'] for p in numeric))}")
print(f"  Degenerate triples (binary only): {sum(1 for p in binary if not p['triangle_valid'])}")
print()
print("Template distribution:")

template_cnt = Counter(p['template_id'] for p in prompts)
def sort_key(t):
    # Sort by type name, then value as string (for consistent ordering)
    return (str(type(t[0])), str(t[0]))
for tid, cnt in sorted(template_cnt.items(), key=sort_key):
    print(f"  template {tid}: {cnt}")
print()
print("First 3 examples:")
for p in prompts[:3]:
    print(f"  [{p['prompt_id']}] task={p.get('task_type','binary')} label={p['label']} tmpl={p['template_id']}")
    print(f"    {p['prompt'][:90]}")

Generated 300 prompts
  Binary : 240
    True : 120
    False: 120
  Numeric: 60
    Target tokens: ['9']
  Degenerate triples (binary only): 24

Template distribution:
  template 1: 57
  template 2: 55
  template 3: 52
  template 4: 52
  template 5: 3
  template 6: 7
  template 7: 9
  template 8: 5
  template N1: 60

First 3 examples:
  [tri_001] task=binary label=False tmpl=1
    There cannot be a triangle with side lengths 8, 8, and 1. Answer:
  [tri_002] task=binary label=True tmpl=6
    For sides 2 and 3, the third side can be 4 (just below maximum). Answer:
  [tri_003] task=binary label=True tmpl=1
    There can be a triangle with side lengths 9, 10, and 12. Answer:


## 2 — Sanity checks

In [3]:
# Verify label_token matches label by task type
for p in prompts:
    task = p.get('task_type', 'binary')
    if task == 'binary':
        expected_tok = TOKEN_TRUE if p['label'] else TOKEN_FALSE
    else:
        expected_tok = f"{p['label']}"
    assert p['label_token'] == expected_tok, f"{p['prompt_id']}: token mismatch"

# Verify binary degenerate fraction ~10%
binary = [p for p in prompts if p.get('task_type', 'binary') == 'binary']
n_degen = sum(1 for p in binary if not p['triangle_valid'])
degen_frac = n_degen / max(len(binary), 1)
assert 0.08 <= degen_frac <= 0.12, f"Binary degenerate fraction {degen_frac:.2%} outside [8%,12%]"

# Verify no duplicate prompt_ids
ids = [p['prompt_id'] for p in prompts]
assert len(ids) == len(set(ids)), "Duplicate prompt_ids found"

# Verify required schema fields
required_fields = {'prompt_id','prompt','task_type','label','label_token','sides',
                   'triangle_valid','template_id','claim_direction'}
for p in prompts:
    missing = required_fields - set(p.keys())
    assert not missing, f"{p['prompt_id']}: missing fields {missing}"

print(f"All sanity checks passed. Binary degenerate fraction: {degen_frac:.1%}")

All sanity checks passed. Binary degenerate fraction: 10.0%


## 3 — Save full dataset

In [4]:
data_dir = MY_WORK / "data"
data_dir.mkdir(parents=True, exist_ok=True)

full_path = data_dir / "prompts_triangle.jsonl"
with open(full_path, "w", encoding="utf-8") as f:
    for p in prompts:
        f.write(json.dumps(p) + "\n")

print(f"Saved {len(prompts)} prompts to {full_path}")

Saved 300 prompts to /workspace/thesis_circuit_breaker/my_work/data/prompts_triangle.jsonl


## 4 — Split into train / eval / analysis

Sizes: **150 train · 75 eval · 75 analysis** (stratified by label).

In [5]:
splits = split_dataset(
    prompts,
    train_frac=0.5,
    eval_frac=0.25,
    analysis_frac=0.25,
    seed=SEED,
)

splits_dir = data_dir / "splits"
splits_dir.mkdir(parents=True, exist_ok=True)

split_file_map = {
    "train":    "train_triangle.jsonl",
    "eval":     "eval.jsonl",
    "analysis": "analysis.jsonl",
}

for split_name, filename in split_file_map.items():
    entries = splits[split_name]
    out_path = splits_dir / filename
    with open(out_path, "w", encoding="utf-8") as f:
        for e in entries:
            f.write(json.dumps(e) + "\n")
    binary_entries = [e for e in entries if e.get('task_type', 'binary') == 'binary']
    numeric_entries = [e for e in entries if e.get('task_type', 'binary') == 'numeric']
    n_t = sum(1 for e in binary_entries if e['label'])
    n_f = sum(1 for e in binary_entries if not e['label'])
    print(
        f"  {split_name:<10} → {out_path.name}  n={len(entries)}  "
        f"(binary={len(binary_entries)}: T={n_t} F={n_f}, numeric={len(numeric_entries)})"
    )

# Verify no overlap between train and eval/analysis
train_ids = {e['prompt_id'] for e in splits['train']}
eval_ids   = {e['prompt_id'] for e in splits['eval']}
ana_ids    = {e['prompt_id'] for e in splits['analysis']}
assert not (train_ids & eval_ids),     "Overlap between train and eval!"
assert not (train_ids & ana_ids),      "Overlap between train and analysis!"
assert not (eval_ids  & ana_ids),      "Overlap between eval and analysis!"
print("\nNo overlap between splits. Dataset ready.")

  train      → train_triangle.jsonl  n=150  (binary=120: T=60 F=60, numeric=30)
  eval       → eval.jsonl  n=75  (binary=60: T=30 F=30, numeric=15)
  analysis   → analysis.jsonl  n=75  (binary=60: T=30 F=30, numeric=15)

No overlap between splits. Dataset ready.


## 5 — Preview analysis split

A quick look at what the attribution loop will process.

In [6]:
analysis = splits['analysis']
binary = [e for e in analysis if e.get('task_type', 'binary') == 'binary']
numeric = [e for e in analysis if e.get('task_type', 'binary') == 'numeric']
print(f"Analysis split: {len(analysis)} prompts")
print(f"  Binary : {len(binary)}")
print(f"    True : {sum(1 for e in binary if e['label'])}")
print(f"    False: {sum(1 for e in binary if not e['label'])}")
print(f"    Degenerate: {sum(1 for e in binary if not e['triangle_valid'])}")
print(f"  Numeric: {len(numeric)}")
print()
print("Sample prompts:")
for e in analysis[:4]:
    print(f"  [{e['prompt_id']}] task={e.get('task_type','binary')} label={e['label']} tmpl={e['template_id']}")
    print(f"    {e['prompt'][:90]}")

Analysis split: 75 prompts
  Binary : 60
    True : 30
    False: 30
    Degenerate: 3
  Numeric: 15

Sample prompts:
  [tri_152] task=binary label=True tmpl=1
    There can be a triangle with side lengths 2, 5, and 6. Answer:
  [tri_224] task=binary label=False tmpl=3
    For a triangle with sides 6 and 8, the third side cannot be 9. Answer:
  [tri_171] task=binary label=False tmpl=1
    There cannot be a triangle with side lengths 8, 15, and 17. Answer:
  [tri_256] task=numeric label=9 tmpl=N1
    For a triangle with sides 9 and 1, what is the largest integer value the third side can ta


## v2 dataset — family × tail factorial design

Generates `prompts_triangle_v2.jsonl` with the supervisor-aligned structure:

| Family | Task | Labels | Tails |
|--------|------|--------|-------|
| `numeric_validity` (~70%) | binary | balanced T/F per tail | 3 |
| `geometry_claim` (~20%) | binary | 6 true + 6 false templates × 3 tails | 3 |
| `numeric_open` (~10%) | numeric | max/min third side | 1 (answer_colon) |

Every row carries `family`, `tail`, `claim_type`, `claim_direction`, `template_id`, `open_kind` so analysis notebooks can slice without rejoining the JSONL.

In [ ]:
## ── v2 dataset generation ─────────────────────────────────────────────────────
# Generates prompts_triangle_v2.jsonl using the supervisor-aligned factorial design:
#   family × tail × balanced binary labels + numeric_open family.
# The v1 file (prompts_triangle.jsonl) is NOT modified.

import importlib
import utils.prompt_generator as pg_mod
importlib.reload(pg_mod)

from utils.prompt_generator import generate_triangle_prompts_v2, summarize_v2

DATASET_V2_FILE = MY_WORK / "data" / "prompts_triangle_v2.jsonl"

prompts_v2 = generate_triangle_prompts_v2(n=300, seed=42)

# Write to disk
import json as _json
DATASET_V2_FILE.parent.mkdir(parents=True, exist_ok=True)
with open(DATASET_V2_FILE, "w", encoding="utf-8") as fh:
    for entry in prompts_v2:
        fh.write(_json.dumps(entry, ensure_ascii=False) + "\n")

print(f"Wrote {len(prompts_v2)} prompts to {DATASET_V2_FILE}\n")
summarize_v2(prompts_v2)

# Verification: check balances
from collections import Counter
binary_v2 = [p for p in prompts_v2 if p["task_type"] == "binary"]
geom_v2   = [p for p in binary_v2 if p["family"] == "geometry_claim"]
num_v2    = [p for p in binary_v2 if p["family"] == "numeric_validity"]
open_v2   = [p for p in prompts_v2 if p["family"] == "numeric_open"]

print(f"\nFamily breakdown: geometry={len(geom_v2)}, numeric_validity={len(num_v2)}, numeric_open={len(open_v2)}")

tail_balance = Counter((p["family"], p["tail"], p["label"]) for p in binary_v2)
print("\nPer-(family, tail) True/False balance:")
print(f"{'family':<22} {'tail':<18} {'label':<8} {'n':>4}")
print("-" * 58)
for (fam, tail, lbl), cnt in sorted(tail_balance.items()):
    print(f"{fam:<22} {tail:<18} {str(lbl):<8} {cnt:>4}")